## Homework 3

> [!NOTE]
> This homework uses the pinned 2026 lead-scoring release in the course
> repository. The plan and report are available in `cohorts/2026/data/`.

### Dataset

In this homework, we will use the 2026 lead-scoring dataset. Download it from [here](https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv).

Or you can do it with `wget`:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
```

In this dataset our desired target for classification task will be `converted` variable - has the client signed up to the platform or not.

### Data preparation

* Check if the missing values are presented in the features.
* If there are missing values:
    * For categorical features, replace them with 'NA'
    * For numerical features, replace with with 0.0 


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


/opt/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv('/Users/foroghakbari/Zoomcamp/ml_course/zoomcamp_ml/classification/course_lead_score.csv')

In [3]:
df.head().T

,0,1,2,3,4
lead_source,organic_search,social_media,referral,paid_ads,referral
industry,technology,technology,retail,manufacturing,manufacturing
employment_status,employed,employed,employed,student,student
location,europe,south_america,europe,NaN,north_america
annual_income,88160.0,72688.0,44697.0,NaN,24062.0
number_of_courses_viewed,3,3,3,3,4
interaction_count,4,7,4,6,5
lead_score,0.64,0.72,0.58,0.57,0.62
converted,1,1,1,0,1


In [4]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [5]:
#fill NAs
# Create a fill values dictionary
fill_dict = {
    col: 0.0 if pd.api.types.is_numeric_dtype(dtype) else 'NA'
    for col, dtype in df.dtypes.items()
}

# Apply fillna in one step
df = df.fillna(value=fill_dict)


In [6]:
# Total missing values per column
print(df.isna().sum())

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


In [7]:
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NA,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [8]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

### Question 1

What is the most frequent observation (mode) for the column `industry`?

- `NA`
- `technology`
- `healthcare`
- `retail`

In [9]:
df['industry']

0          technology
1          technology
2              retail
3       manufacturing
4       manufacturing
            ...      
4995        education
4996       technology
4997          finance
4998       technology
4999       technology
Name: industry, Length: 5000, dtype: object

In [10]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

### Question 2

Create the [correlation matrix](https://www.google.com/search?q=correlation+matrix) for the numerical features of your dataset.
In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

Only consider the pairs above when answering this question.

### Split the data

- Split the data with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
```

- Make sure that the target value `converted` is not in your dataframe.


In [11]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [12]:
# Alternatively, specify exact types:
numeric_df = df.select_dtypes(include=['int', 'float'])

In [13]:
numeric_df

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,88160.0,3,4,0.64,1
1,72688.0,3,7,0.72,1
2,44697.0,3,4,0.58,1
3,0.0,3,6,0.57,0
4,24062.0,4,5,0.62,1
...,...,...,...,...,...
4995,45834.0,2,2,0.37,1
4996,84831.0,2,5,0.51,0
4997,53636.0,4,6,0.75,1
4998,69430.0,4,8,0.79,1


In [14]:
import numpy as np
import pandas as pd

# Your existing code
corr_matrix = df.corr(numeric_only=True)

# Mask the lower triangle and diagonal (self-correlations)
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Stack into a Series, sort by absolute value, and grab top results
top_correlations = (
    upper_triangle.stack()
    .reindex(upper_triangle.stack().abs().sort_values(ascending=False).index)
)

print("Top Correlations:")
print(top_correlations.head(5))

Top Correlations:
interaction_count         lead_score           0.915746
number_of_courses_viewed  lead_score           0.757204
                          interaction_count    0.721609
lead_score                converted            0.483246
interaction_count         converted            0.448624
dtype: float64


In [15]:
df

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
...,...,...,...,...,...,...,...,...,...
4995,organic_search,education,employed,asia,45834.0,2,2,0.37,1
4996,NA,technology,employed,north_america,84831.0,2,5,0.51,0
4997,organic_search,finance,employed,asia,53636.0,4,6,0.75,1
4998,referral,technology,employed,africa,69430.0,4,8,0.79,1


In [16]:
from sklearn.model_selection import train_test_split

In [17]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

Make sure that the target value converted is not in your dataframe.

In [18]:
df_full_train

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
4227,events,healthcare,employed,south_america,47301.0,5,7,0.69,0
4676,organic_search,technology,unemployed,asia,0.0,1,5,0.57,0
800,events,finance,unemployed,asia,47970.0,0,0,0.17,0
3671,organic_search,retail,self_employed,south_america,52847.0,3,3,0.45,0
4193,paid_ads,finance,employed,africa,73460.0,1,4,0.42,1
...,...,...,...,...,...,...,...,...,...
4426,paid_ads,healthcare,self_employed,europe,49442.0,0,3,0.32,1
466,social_media,finance,student,north_america,0.0,1,1,0.23,0
3092,referral,healthcare,student,asia,16378.0,2,5,0.55,1
3772,social_media,NA,NA,north_america,87681.0,2,6,0.76,1


In [19]:
df_train

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
4702,paid_ads,finance,employed,asia,65980.0,2,3,0.37,1
2155,events,finance,employed,south_america,71662.0,0,0,0.09,0
289,referral,NA,employed,asia,40220.0,2,5,0.55,1
1822,organic_search,finance,employed,north_america,88352.0,2,2,0.45,0
3442,events,technology,employed,north_america,89316.0,1,5,0.00,1
...,...,...,...,...,...,...,...,...,...
3200,paid_ads,finance,employed,asia,81522.0,1,3,0.29,0
617,social_media,retail,employed,asia,33075.0,1,1,0.30,1
1992,organic_search,finance,employed,africa,71532.0,3,5,0.58,1
3301,social_media,retail,employed,asia,48836.0,0,0,0.03,0


In [20]:
#as target var is also numeric no need to encode it.

In [21]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [22]:
df_train

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,paid_ads,finance,employed,asia,65980.0,2,3,0.37,1
1,events,finance,employed,south_america,71662.0,0,0,0.09,0
2,referral,NA,employed,asia,40220.0,2,5,0.55,1
3,organic_search,finance,employed,north_america,88352.0,2,2,0.45,0
4,events,technology,employed,north_america,89316.0,1,5,0.00,1
...,...,...,...,...,...,...,...,...,...
2995,paid_ads,finance,employed,asia,81522.0,1,3,0.29,0
2996,social_media,retail,employed,asia,33075.0,1,1,0.30,1
2997,organic_search,finance,employed,africa,71532.0,3,5,0.58,1
2998,social_media,retail,employed,asia,48836.0,0,0,0.03,0


In [23]:
## target variable

In [24]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [25]:
y_train

array([1, 0, 1, ..., 1, 0, 0])

In [26]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

### Question 3 - EDA

- Calculate the mutual information score between `converted` and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using `round(score, 2)`.

Which of these variables has the biggest mutual information score?

- `industry`
- `location`
- `lead_source`
- `employment_status`


In [27]:
df_full_train = df_full_train.reset_index(drop=True)

In [28]:
df_full_train.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [29]:
## at the target variable 

In [30]:
df_full_train.converted.value_counts(normalize=True)

converted
1    0.57075
0    0.42925
Name: proportion, dtype: float64

In [31]:
df_full_train.converted.mean()

0.57075

In [32]:
#numerical features

In [33]:
numeric_df = df.select_dtypes(include=['int', 'float'])

In [34]:
numeric_df.dtypes

annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [35]:
numeric = ['annual_income', 'number_of_courses_viewed','interaction_count','lead_score']

categorical = ['lead_source', 'industry','employment_status','location']

In [36]:
df_full_train[categorical].nunique()


lead_source          6
industry             7
employment_status    5
location             6
dtype: int64

In [37]:
df_full_train[numeric].nunique()


annual_income               3469
number_of_courses_viewed       7
interaction_count             14
lead_score                   100
dtype: int64

Churn rate: Difference between global mean of the target variable and mean of the target variable for categories of a feature. If this difference is greater than 0, it means that the category is less likely to churn, and if the difference is lower than 0, the group is more likely to churn. The larger differences are indicators that a variable is more important than others.

In [38]:
global_converted = df_full_train.converted.mean()

In [39]:
print(global_converted)

0.57075


In [40]:
#with other features 

In [41]:
df_full_train['employment_status'].unique()

array(['employed', 'unemployed', 'self_employed', 'student', 'NA'],
      dtype=object)

In [42]:
converted_employed = df_full_train[df_full_train.employment_status == 'employed'].converted.mean()
converted_unemployed = df_full_train[df_full_train.employment_status == 'unemployed'].converted.mean()

In [43]:
converted_employed

0.6564593301435406

In [44]:
converted_unemployed

0.4355716878402904

In [45]:
global_converted - converted_employed

-0.08570933014354065

In [46]:
global_converted - converted_unemployed

0.1351783121597096

Risk ratio: Ratio between mean of the target variable for categories of a feature and global mean of the target variable. If this ratio is greater than 1, the category is more likely to churn, and if the ratio is lower than 1, the category is less likely to churn. It expresses the feature importance in relative terms.

In [47]:
from sklearn.metrics import mutual_info_score

In [48]:
mutual_info_score(df_full_train.converted, df_full_train.industry)

0.0028984818739526547

In [49]:
df_full_train

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,events,healthcare,employed,south_america,47301.0,5,7,0.69,0
1,organic_search,technology,unemployed,asia,0.0,1,5,0.57,0
2,events,finance,unemployed,asia,47970.0,0,0,0.17,0
3,organic_search,retail,self_employed,south_america,52847.0,3,3,0.45,0
4,paid_ads,finance,employed,africa,73460.0,1,4,0.42,1
...,...,...,...,...,...,...,...,...,...
3995,paid_ads,healthcare,self_employed,europe,49442.0,0,3,0.32,1
3996,social_media,finance,student,north_america,0.0,1,1,0.23,0
3997,referral,healthcare,student,asia,16378.0,2,5,0.55,1
3998,social_media,NA,NA,north_america,87681.0,2,6,0.76,1


In [50]:
def mutual_info_conv_score(series):
    return round(mutual_info_score(series, df_full_train.converted), 2)

In [51]:
mi = df_full_train[categorical].apply(mutual_info_conv_score)
mi.sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [52]:
#lead score is the most important 

### Question 4

- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
  - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
  - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.55
- 0.65
- 0.75
- 0.85




One-hot encoding- converted our cateogories into dummy variables 0,1

In [53]:
from sklearn.feature_extraction import DictVectorizer

In [54]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numeric].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numeric].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [55]:
def logistic_regression(xi):
    score = w0
    
    for j in range(len(w)):
        score = score + xi[j] * w[j]
        
    result = sigmoid(score)
    return result

In [61]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='lbfgs')


In [62]:
model.fit(X_train, y_train)

LogisticRegression()

In [63]:
model.intercept_[0]

-0.010726542284659793

In [64]:
model.coef_[0].round(3)

array([-0.   , -0.001,  0.01 , -0.004, -0.009, -0.007, -0.   , -0.003,
       -0.001, -0.002, -0.002, -0.001, -0.001,  0.203,  0.016,  0.   ,
       -0.011,  0.01 , -0.017,  0.012, -0.005,  0.   , -0.002, -0.001,
        0.   , -0.004, -0.004,  0.08 ])

In [65]:
y_pred = model.predict_proba(X_val)[:, 1]

In [66]:
convert_decision = (y_pred >= 0.5)

In [67]:
(y_val == convert_decision).mean()

0.658

In [68]:
df_pred = pd.DataFrame()
df_pred['probability'] = y_pred
df_pred['prediction'] = convert_decision.astype(int)
df_pred['actual'] = y_val

In [69]:

df_pred['correct'] = df_pred.prediction == df_pred.actual

In [70]:
df_pred.correct.mean()

0.658

In [71]:
convert_decision.astype(int)

array([1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1,
       0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1,
       1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1,

### Question 5

- Let's find the least useful feature using the _feature elimination_ technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- `'lead_source'`
- `'number_of_courses_viewed'`
- `'interaction_count'`

> **Note**: The difference doesn't have to be positive.



### Question 6

- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter `C`: `[0.000001, 0.00001, 0.0001, 0.001]`.
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001
